# 📊 WRDS Tech Stock Analysis Project

## Student Information
- Name: Bingbing Zhang  
- Student ID:2469540
- Course: ACC102 Data Product  
- Date: April 2026  

---

## Objective

This project uses WRDS (CRSP database) to analyze major technology stocks and evaluate their risk-adjusted performance.

The main question is:

> Which tech stock provides the best balance between return and risk?

## Data Source

The dataset is obtained from WRDS CRSP (Center for Research in Security Prices), which is widely used in academic finance research.

It contains:
- Daily stock returns
- Stock identifiers (permno)
- Time period: 2020–present

## Data Description

The dataset includes daily stock return observations.

Each record contains:
- Date
- Permno (unique stock identifier)
- Daily return (ret)

This structure allows computation of long-term performance and risk metrics.

In [1]:
import wrds
import pandas as pd
import numpy as np

db = wrds.Connection()

Enter your WRDS username [zhangbingbing]: effortless
Enter your password: ········


WRDS recommends setting up a .pgpass file.


Create .pgpass file now [y/n]?:  y


Created .pgpass file successfully.
You can create this file yourself at any time with the create_pgpass_file() function.
Loading library list...
Done


## Data Extraction

We extract stock return data using SQL from CRSP database.

In [2]:
query = """
SELECT date, permno, ret
FROM crsp.dsf
WHERE permno IN (14593, 10107, 84788)
AND date >= '2020-01-01'
"""

df = db.raw_sql(query)
df.head()

,date,permno,ret
0,2020-01-02,10107,0.018516
1,2020-01-03,10107,-0.012452
2,2020-01-06,10107,0.002585
3,2020-01-07,10107,-0.009118
4,2020-01-08,10107,0.015928


## Data Cleaning

We convert return values to numeric format and remove missing values.

In [3]:
df['ret'] = pd.to_numeric(df['ret'], errors='coerce')
df = df.dropna()

## Feature Engineering

We calculate three key financial metrics:

- Annualized Return (growth)
- Volatility (risk)
- Sharpe Ratio (risk-adjusted return)

In [4]:
results = []

for permno in df['permno'].unique():
    sub = df[df['permno'] == permno]

    ann_return = sub['ret'].mean() * 252
    volatility = sub['ret'].std() * np.sqrt(252)

    sharpe = ann_return / volatility if volatility != 0 else np.nan

    results.append({
        "permno": permno,
        "Return": ann_return,
        "Volatility": volatility,
        "Sharpe": sharpe
    })

result_df = pd.DataFrame(results)
result_df

,permno,Return,Volatility,Sharpe
0,10107,0.252452,0.304947,0.827855
1,14593,0.302222,0.316803,0.953975
2,84788,0.237909,0.359688,0.661431


## Mapping Companies

We map CRSP permno identifiers to real company names.

In [5]:
mapping = {
    14593: "AAPL",
    10107: "MSFT",
    84788: "GOOGL"
}

result_df["Company"] = result_df["permno"].map(mapping)
result_df

,permno,Return,Volatility,Sharpe,Company
0,10107,0.252452,0.304947,0.827855,MSFT
1,14593,0.302222,0.316803,0.953975,AAPL
2,84788,0.237909,0.359688,0.661431,GOOGL


## Methodology

Three key financial indicators are used:

- Annualized Return: measures long-term growth
- Volatility: measures risk level
- Sharpe Ratio: measures risk-adjusted return

These metrics are widely used in academic finance and portfolio analysis.

## Analysis

We evaluate stock performance based on:

- High return → growth potential
- Low volatility → stability
- High Sharpe ratio → best risk-adjusted performance

In [6]:
result_df["return_score"] = result_df["Return"].rank(pct=True)
result_df["risk_score"] = 1 - result_df["Volatility"].rank(pct=True)
result_df["sharpe_score"] = result_df["Sharpe"].rank(pct=True)

result_df["total_score"] = (
    0.4 * result_df["return_score"] +
    0.3 * result_df["risk_score"] +
    0.3 * result_df["sharpe_score"]
)

result_df.sort_values("total_score", ascending=False)

,permno,Return,Volatility,Sharpe,Company,return_score,risk_score,sharpe_score,total_score
1,14593,0.302222,0.316803,0.953975,AAPL,1.000000,0.333333,1.000000,0.800000
0,10107,0.252452,0.304947,0.827855,MSFT,0.666667,0.666667,0.666667,0.666667
2,84788,0.237909,0.359688,0.661431,GOOGL,0.333333,0.000000,0.333333,0.233333


## Sharpe Ratio Explanation

Sharpe Ratio is defined as:

> Sharpe = Return / Volatility

It measures how much return is generated per unit of risk.

A higher Sharpe Ratio indicates better risk-adjusted performance.

## Final Interpretation

The best-performing stock is selected based on the highest total score.

This indicates the optimal balance between:
- Strong growth (Return)
- Low risk (Volatility)
- Efficient performance (Sharpe Ratio)

## Visualization

The ranking results help visually compare the risk-adjusted performance of different stocks.

This makes it easier to identify the best investment choice.

## Limitations

- Limited number of stocks included
- Simplified financial model
- No macroeconomic factors considered
- Data depends on availability in WRDS CRSP

## Conclusion

This project demonstrates how WRDS data can be used to evaluate stock performance using academic financial metrics.

The final model identifies the best stock based on a combination of return, risk, and Sharpe ratio.